# AniNow Klima — what the model says, and how far it can be trusted

Rice farmers in the Philippines hold a field for roughly **197 days** between the
November harvest and the next planting. The question this model answers is what, if
anything, is worth putting in it: a second rice crop, or one of 28 alternatives.

It answers by **replaying 44 past seasons** of NASA POWER weather through published
agronomic limits. It is not a forecast and it is not trained on anything.

Run `model/klima_model.py` and `model/analysis.py --out docs/analysis` before this
notebook: every figure below is drawn from what they write, never recomputed here.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import make_figures as F

data = F.load()
seasons, places, crops, pairs, results = data
print(f'{len(places)} places, {len(crops)} crops, {len(seasons):,} crop-seasons')
print(f"pass mark {results['meta']['pass_mark']} | window {results['meta']['window_days']} days")

## 1. How a recommendation is reached

Six steps. Hard feasibility first, then scores, then a count — never the other way
round, because a crop that cannot physically fit the window should not be scored at
all.

In [ ]:
F.fig_how_it_decides(data)[0]

## 2. The model, as a function

Four conditions are each scored 0 to 1, and the season's suitability is the **weakest**
of them:

$$S = \min\bigl(\text{temperature},\ \text{water},\ \text{soil pH},\ \text{soil texture}\bigr)$$

The minimum, not an average, because a crop is only as well served as the thing it is
shortest of — Liebig's law of the minimum, which is also the rule EcoCrop's own model
uses. An average would let good soil pay for absent water.

Water is the only condition that changes from season to season:

$$W = \min\left(1,\ \frac{\text{rain over the cycle}}{ET_c}\right),
\qquad ET_c = \sum_{i} K_c(i)\, ET_0(i) + n \cdot P_{\text{seepage}}$$

with reference evapotranspiration from FAO-56 Eq. 52 (Hargreaves):

$$ET_0 = 0.0023\,(T_{\text{mean}} + 17.8)\,\sqrt{T_{\max} - T_{\min}}\;R_a$$

A season is a **good year** when $S \ge 0.6$. Both curves below are published; the 0.6
is ours. Full derivation in [docs/Model-Mathematics.md](../docs/Model-Mathematics.md).

In [ ]:
F.fig_the_function(data)[0]

## 3. When to plant

The biggest lever a farmer actually controls is the planting date, so the whole curve
is reported rather than a hidden best pick.

In [ ]:
F.fig_when_to_plant(data)[0]

At the pilot field a second rice crop is short by about **964 mm** in a typical
season. Mungbean is short by **44 mm**. That gap, not a yield model, is what the
recommendation rests on.

Rice's figure is an **underestimate**: FAO Training Manual 3's saturation (200 mm) and
standing water layer (100 mm) are not included.

## 4. Forty-four seasons, replayed

Each point is one real past season, scored on that season's own weather.

In [ ]:
F.fig_seasons(data)[0]

## 5. What actually decides the answer

Because $S$ is a minimum, it reports only the binding condition. If one condition bound
every pair, the model would be a single-variable calculator wearing four names.

In [ ]:
F.fig_what_decides(data)[0]

## 6. Across 22 places

Same questions at every place: harvest 15 November, next planting 31 May, rainfed,
field drains, rice grown now.

In [ ]:
fig, _ = F.fig_places(data); fig

In [ ]:
fig, _ = F.fig_water_saved(data); fig

## 7. The constants we chose, not the sources

The model has **no fitted parameters**, but it does have chosen ones, and the pass mark
is the most consequential. No source sets it.

In [ ]:
F.fig_pass_mark(data)[0]

## 8. Two things that weaken the statistics

Reported here rather than left for a reader to find.

**The record is not one climate.** Window rainfall rises at 21 of 22 places
(Mann-Kendall, Benjamini-Hochberg at 5% FDR), so a count pooled over 41 years
understates the recent half. This is a trend in the NASA POWER reanalysis; it has not
been checked against a rain gauge and is not evidence of climate change.

**El Niño is underpowered.** The direction is consistent, the proof is not.

In [ ]:
fig, _ = F.fig_halves(data); fig

In [ ]:
fig, _ = F.fig_el_nino(data); fig

## What these figures cannot support

- **Accuracy, precision, recall, or any claim that a recommendation was right.**
  There is no harvest record for these fields to test against, so no such figure
  exists and none is reported.
- A forecast for next season. Every number here is a count or a median over past
  seasons.
- Yields, income or prices. The model has none.
- "Covers the Philippines". It covers the 22 places listed, in 15 distinct weather
  cells — places sharing a cell are not independent tests of the climate.

What *is* measured, and how, is set out in
[docs/analysis/pitch_stats.md](../docs/analysis/pitch_stats.md).

In [ ]:
for fn in F.FIGURES:
    fig, name = fn(data)
    fig.savefig(os.path.join('..', 'docs', 'figures', name + '.png'))
    F.plt.close(fig)
print(f'{len(F.FIGURES)} figures written to docs/figures/')